In [1]:
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq


In [2]:

# reproducibility
rng = np.random.default_rng(42)

# configuration
classes = 5
points_per_class = 200_000_000

# class sizes (majority/minority imbalance)
class_sizes = [
    int(points_per_class * 1.5),  # majority
    points_per_class,
    points_per_class,
    int(points_per_class * 0.5),  # minority
    int(points_per_class * 0.25)  # strong minority
]

# gaussian parameters (means and stds)
params = [
    {"mean": [-0.6, -0.6], "std": [0.20, 0.20]},
    {"mean": [-0.4, -0.5], "std": [0.30, 0.25]},
    {"mean": [-0.2, 0.5], "std": [0.25, 0.25]},
    {"mean": [0.6, 0.6], "std": [0.18, 0.20]},
    {"mean": [0.5, 0.0], "std": [0.40, 0.40]},  # overlapping center class
]

all_data = []

for class_id in range(classes):

    n = class_sizes[class_id]
    mean = params[class_id]["mean"]
    std = params[class_id]["std"]

    points = rng.normal(loc=mean, scale=std, size=(n, 2)).astype(np.float32)

    # clip into [-1,1] range
    points = np.clip(points, -1, 1)

    df = pd.DataFrame({
        "x": points[:, 0],
        "y": points[:, 1],
        "label": np.full(n, class_id, dtype=np.int8)
    })

    all_data.append(df)

data = pd.concat(all_data, ignore_index=True)

# convert to Arrow table
table = pa.Table.from_pandas(data)

# save parquet
pq.write_table(
    table,
    "gaussian_overlap_dataset.parquet",
    compression="zstd"
)

print("Saved dataset:", data.shape)

Saved dataset: (850000000, 3)
